# ResQAI: Sri Lanka Disaster Hazard & Flood Susceptibility Prediction

This notebook explores machine learning models for predicting **flood occurrence** and **multi-hazard risk** across the **25 administrative districts of Sri Lanka**.

### Key Sri Lankan Climate & Disaster Drivers:
- **Southwest Monsoon (SWM / Yala)**: May to September — heavy rainfall in the Western, Sabaragamuwa, and Southern wet zones.
- **Northeast Monsoon (NEM / Maha)**: December to February — heavy rainfall in Eastern, Northern, and North-Central dry zones.
- **Second Intermonsoon**: October to November — island-wide intense convective rainfall triggering flash floods and landslides.
- **Primary River Basins**: Kelani Ganga, Kalu Ganga, Gin Ganga, Nilwala Ganga, Mahaweli Ganga.
- **Landslides**: Regulated by **NBRO** (National Building Research Organisation) rainfall thresholds (>75mm, >100mm, >150mm).

In [ ]:
from typing import cast, Any
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load Sri Lanka Datasets
df_districts = pd.read_csv("../datasets/sri_lanka_districts.csv")
df_history = pd.read_csv("../datasets/sri_lanka_flood_landslide_history.csv")
print(f"Loaded {len(df_districts)} districts and {len(df_history)} historical records.")
df_districts.head()

## 1. Exploratory Data Analysis (EDA)
Let's examine the distribution of flood occurrences and rainfall across Sri Lankan climatic zones.

In [ ]:
print("Distribution of Flood Severity:")
print(df_history["FLOOD_OCCURRENCE"].value_counts())

print("\nDistribution of NBRO Landslide Alerts:")
print(df_history["NBRO_LANDSLIDE_ALERT"].value_counts())

# Average Annual Rainfall by Climatic Zone
zone_summary = df_history.groupby("ZONE")["ANNUAL"].mean().round(1)
print("\nAverage Annual Rainfall (mm) by Zone:")
print(zone_summary)

## 2. Feature Preprocessing & District Encoding

In [ ]:
district_encoder = LabelEncoder()
df_history["DISTRICT_ENCODED"] = cast(Any, district_encoder.fit_transform(df_history["DISTRICT"]))

feature_cols = [
    "DISTRICT_ENCODED",
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC",
    "ANNUAL", "PEAK_24H_RAIN"
]

X = df_history[feature_cols]
y_flood = df_history["FLOOD_OCCURRENCE"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_flood, test_size=0.20, random_state=42, stratify=y_flood
)
print(f"Training set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

## 3. Model Training & Comparison

In [ ]:
# Decision Tree Baseline
dt_model = DecisionTreeClassifier(max_depth=6, random_state=42)
dt_model.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt_model.predict(X_test))
print(f"Decision Tree Accuracy: {dt_acc * 100:.2f}%")

# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf_model.predict(X_test))
print(f"Random Forest Accuracy: {rf_acc * 100:.2f}%")

print("\nClassification Report (Random Forest):")
print(classification_report(y_test, rf_model.predict(X_test)))

## 4. Feature Importance Analysis

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top 5 Predictive Features:")
print(importances.head(5))

## 5. Testing Real-World Sri Lankan Scenarios

In [ ]:
# Example test on Colombo during extreme Southwest Monsoon depression
colombo_encoded = cast(Any, district_encoder.transform(["Colombo"]))[0]
sample_case = pd.DataFrame([{
    "DISTRICT_ENCODED": colombo_encoded,
    "JAN": 45.0, "FEB": 30.0, "MAR": 70.0, "APR": 140.0,
    "MAY": 480.0, "JUN": 420.0, "JUL": 290.0, "AUG": 180.0,
    "SEP": 240.0, "OCT": 310.0, "NOV": 290.0, "DEC": 120.0,
    "ANNUAL": 2615.0, "PEAK_24H_RAIN": 185.0
}])

pred = rf_model.predict(sample_case)[0]
probs = dict(zip(rf_model.classes_, rf_model.predict_proba(sample_case)[0]))
print(f"Prediction for Colombo: {pred}")
print(f"Probabilities: {probs}")